# Tools Definition

In [37]:
from dotenv import load_dotenv

load_dotenv()

True

In [38]:
from langchain.tools import tool

@tool
def square_root(num: float) -> float:
    "This function takes the number as input and produces Square root of that number as output."
    return num ** 0.5

In [39]:
# tools can be defined in other ways
@tool("square_root")
def tool1(num: float) -> float:
    "This function takes the number as input and produces Square root of that number as output."
    return num ** 0.5


@tool("square_root", description= "This function takes the number as input and produces Square root of that number as output.")
def tool2(num: float) -> float:
    return num ** 0.5

In [40]:
# for calling
tool1.invoke({"num":420})

20.493901531919196

# Adding to Agents

In [41]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],
    system_prompt="You're an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

question = HumanMessage(content="What is the square root of 619?")

response = agent.invoke({
    "messages":[question]
})

response["messages"]

[HumanMessage(content='What is the square root of 619?', additional_kwargs={}, response_metadata={}, id='0e2d1ebb-79e5-4660-92bf-9c7bb9eab407'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2839, 'prompt_tokens': 166, 'total_tokens': 3005, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2816, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DaCY1oQww0baYufZiEgE09BjceRqY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ddc79-0a21-7531-90fc-69ff1e2b1509-0', tool_calls=[{'name': 'square_root', 'args': {'num': 619}, 'id': 'call_hYWtDrOjwlhaM4CzsfUxZZBT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 2839, 'tot

In [42]:
from pprint import pprint


print(response["messages"][-1].content)

pprint(response["messages"])

√619 ≈ 24.879710609249457

If you’d like a shorter rounding, it’s about 24.8797.
[HumanMessage(content='What is the square root of 619?', additional_kwargs={}, response_metadata={}, id='0e2d1ebb-79e5-4660-92bf-9c7bb9eab407'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2839, 'prompt_tokens': 166, 'total_tokens': 3005, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2816, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DaCY1oQww0baYufZiEgE09BjceRqY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ddc79-0a21-7531-90fc-69ff1e2b1509-0', tool_calls=[{'name': 'square_root', 'args': {'num': 619}, 'id': 'call_hYWtDrOjwlhaM4CzsfUxZZBT', 'type': 'tool_call'}], invali

In [43]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'num': 619}, 'id': 'call_hYWtDrOjwlhaM4CzsfUxZZBT', 'type': 'tool_call'}]


# Without Web search

In [44]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",    
)

question = HumanMessage(content="How up to date is your knowledge?")

response = agent.invoke({
    "messages": [question]
})

print(response["messages"])



[HumanMessage(content='How up to date is your knowledge?', additional_kwargs={}, response_metadata={}, id='a2604630-7919-4de3-8ea9-bc9a9b275c20'), AIMessage(content='Knowledge cutoff is June 2024. I don’t have real-time data by default, and I don’t know about events or releases after that unless you provide them or the chat environment has browsing enabled.\n\nWhat this means:\n- I can help with information and analysis up to mid-2024.\n- For anything after that, I can still reason about concepts, trends, and prior context, or you can supply the latest details.\n- If your platform supports browsing and you enable it, I can look up current information. Otherwise, I’ll rely on what you share or on general guidance to verify.\n\nIf you have a specific topic or event in mind, tell me and I’ll tailor the answer to the latest info I was trained on or help you verify with sources.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 932, 'prompt_token

In [45]:
print(response["messages"][-1].content)

Knowledge cutoff is June 2024. I don’t have real-time data by default, and I don’t know about events or releases after that unless you provide them or the chat environment has browsing enabled.

What this means:
- I can help with information and analysis up to mid-2024.
- For anything after that, I can still reason about concepts, trends, and prior context, or you can supply the latest details.
- If your platform supports browsing and you enable it, I can look up current information. Otherwise, I’ll rely on what you share or on general guidance to verify.

If you have a specific topic or event in mind, tell me and I’ll tailor the answer to the latest info I was trained on or help you verify with sources.


# Add Web search tool

In [46]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> str:
    """ Search the web for information """
    return tavily_client.search(query=query)


web_search.invoke("Who is the current President of India?")


{'query': 'Who is the current President of India?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/President_of_India',
   'title': 'President of India - Wikipedia',
   'content': 'Droupadi Murmu is the 15th and current president, having taken office on 25 July 2022. President of India. Bhārata kē Rāṣṭrapati. State emblem · f. Flag of',
   'score': 0.99946004,
   'raw_content': None},
  {'url': 'http://www.presidentofindia.gov.in/',
   'title': 'President of India: Home',
   'content': 'Smt. Droupadi Murmu was sworn in as the 15th President of India on 25 July, 2022. Previously, she was the Governor of Jharkhand from 2015 to 2021.',
   'score': 0.9978002,
   'raw_content': None},
  {'url': 'http://www.presidentofindia.gov.in/Profile',
   'title': 'Profile | President of India',
   'content': 'Smt. Droupadi Murmu was sworn in as the 15th President of India on July 25, 2022. Prior to this, she served as the 9th Governor o

In [47]:
agent = create_agent(
    model = "gpt-5-nano",
    tools=[web_search]
)

question = HumanMessage(content="Who is the current President of India?")

response = agent.invoke(
    {"messages":[question]}
)


response["messages"][-1].content

'Droupadi Murmu is the current President of India (the 15th President). She was sworn in on 25 July 2022. If you’d like, I can pull up the latest official confirmation or provide more details about her tenure.'

In [48]:
pprint(response["messages"])

[HumanMessage(content='Who is the current President of India?', additional_kwargs={}, response_metadata={}, id='0d46d4dc-ef40-4816-bfda-f45d37f1242a'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 418, 'prompt_tokens': 132, 'total_tokens': 550, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DaCYeKiyIyQTr0sdTQVrzfde7ZYcr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ddc79-a0e7-71f1-977f-5158cf3c3099-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current President of India 2026 Droupadi Murmu'}, 'id': 'call_TkKkuYwXLKHL26UYImznGO3D', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata